## Wikipedia Retriever


In [1]:
from langchain_community.retrievers import WikipediaRetriever
from dotenv import load_dotenv
import wikipedia
load_dotenv()

True

In [2]:
# The WikipediaRetriever in langchain_community relies on an underlying third-party wikipedia Python package. This library sends requests to the
# Wikimedia API using a generic, unmaintained default User-Agent string.
# Because Wikimedia blocks or severely rate-limits requests containing this default User-Agent string, the Wikipedia server refuses to process
# the query and returns an empty or HTML-formatted error block instead of valid JSON. When LangChain tries to decode this response via
# response.json(), it crashes with Expecting value: line 1 column 1 (char 0)
wikipedia.set_user_agent(
    "MyRAGApplication/1.0 (contact: your_email.com)")

In [3]:
# Initialize the retriever (optional: set language and top_k)
# top_k_results are the no of results we want from the retriever
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [4]:
# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [5]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:")
    print(f"{doc.page_content}")


Result 1:
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear advantage, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,000 Pakistani ser

## Vector Store Retriever


In [6]:
from langchain_chroma import Chroma
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [7]:
# Step 1: Your source documents
document = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(
        page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [10]:
# Step 2: Initialize embedding model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")

In [12]:
# Step 3: Create Chroma vector store in memory
vector_store = Chroma.from_documents(
    documents=document,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [14]:
# Step 4: Convert vectorstore into a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

In [15]:
query = "What is Chroma used for"
results = retriever.invoke(query)

In [16]:
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(f"{doc.page_content}\n")


Result 1
Chroma is a vector database optimized for LLM-based search.


Result 2
Chroma is a vector database optimized for LLM-based search.



## MMR


In [17]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(
        page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [19]:
from langchain_community.vectorstores import FAISS

model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=model
)

In [20]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    # k = top results, lambda_mult = relevance-diversity balance
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)  # lambda_mult -> it is factor that varies the result 0-> MMR and 1-> normal semantic search

In [21]:
query = "what is langchain"

In [22]:
results = retriever.invoke(query)

In [23]:
# lambda_mult": 1 (normal semantic search) it gives simillar types of results
for i, doc in enumerate(results):
    print(f"Result {i+1}")
    print(f"{doc.page_content}\n")

Result 1
LangChain is used to build LLM based applications.

Result 2
MMR helps you get diverse results when doing similarity search.

Result 3
LangChain supports Chroma, FAISS, Pinecone, and more.



In [24]:
# lambda_mult": 0.5 (MMR) it gives distinct and diverse results
for i, doc in enumerate(results):
    print(f"Result {i+1}")
    print(f"{doc.page_content}\n")

Result 1
LangChain is used to build LLM based applications.

Result 2
MMR helps you get diverse results when doing similarity search.

Result 3
LangChain supports Chroma, FAISS, Pinecone, and more.



## MultiQuery Retrievers


In [26]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [27]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.",
             metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.",
             metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.",
             metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.",
             metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.",
             metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.",
             metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.",
             metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.",
             metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.",
             metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.",
             metadata={"source": "I5"}),
]

In [28]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")

vector_store = FAISS.from_documents(
    embedding=embedding_model,
    documents=all_docs
)

In [29]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [30]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")
)

In [31]:
query = "How to improve energy levels and maintain balance?"

In [32]:
similarity_res = retriever.invoke(query)

In [33]:
for i, doc in enumerate(similarity_res):
    print(f"Result {i+1}")
    print(f"{doc.page_content}\n")

Result 1
Drinking sufficient water throughout the day helps maintain metabolism and energy.

Result 2
The solar energy system in modern homes helps balance electricity demand.

Result 3
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

Result 4
Consuming leafy greens and fruits helps detox the body and improve longevity.

Result 5
Regular walking boosts heart health and can reduce symptoms of depression.



In [34]:
multiquery_result = multiquery_retriever.invoke(query)

In [35]:
for i, doc in enumerate(multiquery_result):
    print(f"Result {i+1}")
    print(f"{doc.page_content}\n")

Result 1
Drinking sufficient water throughout the day helps maintain metabolism and energy.

Result 2
The solar energy system in modern homes helps balance electricity demand.

Result 3
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

Result 4
Consuming leafy greens and fruits helps detox the body and improve longevity.

Result 5
Regular walking boosts heart health and can reduce symptoms of depression.

Result 6
Deep sleep is crucial for cellular repair and emotional regulation.



## ContextualCompressionRetriever


In [36]:
from langchain_community.vectorstores import FAISS
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [37]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [38]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")

vector_store = FAISS.from_documents(
    embedding=embedding_model,
    documents=docs
)

In [39]:
base_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [40]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")
compressor = LLMChainExtractor.from_llm(llm)

In [41]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [42]:
query = "What is photosynthesis?"
result = compression_retriever.invoke(query)

In [43]:
for i, doc in enumerate(result):
    print(f"Result {i+1}")
    print(f"{doc.page_content}\n")

Result 1
Photosynthesis is the process by which green plants convert sunlight into energy.

Result 2
The chlorophyll in plant cells captures sunlight during photosynthesis.

